In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
import os
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from torchvision.transforms import InterpolationMode
import torchvision.transforms.functional as TF
import time
from tqdm import tqdm
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import cv2
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D




In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  MODE SELECTOR
#  Valid values:
#    's1only'    – train Stage 1 only
#    's2only'    – train Stage 2 only (needs S1 weights)
#    's1s2'      – train Stage 1, then Stage 2
#    'both'      – full pipeline
#    'eval_s1'   – test only Stage 1 
#    'eval_s2'   – test only Stage 2 (needs S1 weights)
#    'eval_all'  – evaluate both stages (needs S1 & S2 weights)
# ══════════════════════════════════════════════════════════════════════
MODE = 'both'

# path for models (stage1 & stage2)
# S1_WEIGHTS_PATH  : path for trained stage1 model weights
# S2_WEIGHTS_PATH  : path for trained stage2 model weights


# S1_WEIGHTS_PATH = "/kaggle/input/datasets/ravat1920/lim-cd-model/Group15_LIM-CD_stage1.pth"
# S2_WEIGHTS_PATH = "/kaggle/input/datasets/ravat1920/lim-cd-model/Group15_LIM-CD_stage2.pth"


S1_WEIGHTS_PATH = "/kaggle/input/datasets/arpitiitk/levir-cd-model/Group15_LEVIR-CD_stage1(apr21).pth"
S2_WEIGHTS_PATH = "/kaggle/input/datasets/arpitiitk/levir-model-s2/Group15_LEVIR-CD_stage2(apr21).pth"




#  Resume from mid-training checkpoints
RESUME_S1_CKPT = None   
RESUME_S2_CKPT = None


_m = MODE
RUN_S1_TRAIN = _m in ('s1only', 's1s2','both')
RUN_S2_TRAIN = _m in ('s2only', 's1s2', 'both')

EVAL_S1 = _m in ('eval_s1', 'eval_all') or RUN_S1_TRAIN
EVAL_S2 = _m in ('eval_s2', 'eval_all') or RUN_S2_TRAIN


print(f'MODE        : {MODE}')
print(f'Train S1    : {RUN_S1_TRAIN}')
print(f'Train S2    : {RUN_S2_TRAIN}')

print(f'Eval  S1    : {EVAL_S1}')
print(f'Eval  S2    : {EVAL_S2}')


In [ ]:
scaler = torch.amp.GradScaler('cuda')

device = "cuda" if torch.cuda.is_available() else "cpu"


SHOW_PROGRESS = False

NUM_EPOCHS_S1 =50
# NUM_EPOCHS_S1 =1
UNFREEZE_EPOCH       = 5
EARLY_STOP_PATIENCE  = 15
CHECKPOINT_EVERY     = 20

NUM_EPOCHS_S2 =45
# NUM_EPOCHS_S2 =1
EARLY_STOP_PATIENCE_S2 = 15
CHECKPOINT_EVERY_S2  = 10



print(f"Device      : {device}")
if RUN_S1_TRAIN:
    print(f"S1 Epochs   : {NUM_EPOCHS_S1}")
if RUN_S2_TRAIN:
    print(f"S2 Epochs   : {NUM_EPOCHS_S2}")


In [ ]:
# Dataset configurations


DATASET = "LIM-CD"   # "LEVIR-CD" or "LIM-CD"

if DATASET == "LEVIR-CD":
    ROOT = "/kaggle/input/datasets/mdrifaturrahman33/levir-cd/LEVIR CD"
    DIRMAP  = {"A": "A", "B": "B", "label": "label"}
    TILESIZE = 256     # LEVIR native=1024, must patch to 256
    STRIDE   = 256       
    BATCH_SIZE = 8
    POSWEIGHT    = 2.5
    MIN_CHANGE   = 10
    FOCAL_ALPHA  = 0.25
    DROPOUT_P   = 0.0 

elif DATASET == "LIM-CD":
    ROOT = "/kaggle/input/datasets/ravat1920/lim-cd/lim-cd"
    DIRMAP  = {"A": "t1", "B": "t2", "label": "label"}
    TILESIZE = 512     
    STRIDE   = 512
    BATCH_SIZE = 4    
    POSWEIGHT    = 1.5   
    MIN_CHANGE   = 40     
    FOCAL_ALPHA  = 0.5    
    DROPOUT_P   = 0.2


print("Dataset :", DATASET)       
print("Root    :", ROOT)
print("Mapping :", DIRMAP)         
print(f"TileSize : {TILESIZE}  Stride : {STRIDE}  Batch : {BATCH_SIZE}")
print(f"PosWeight: {POSWEIGHT}  MinChange: {MIN_CHANGE}  FocalAlpha: {FOCAL_ALPHA}")

In [ ]:
MODEL_NAME = "Group15"
MODEL_PATH = f"{MODEL_NAME}_{DATASET}".replace(" ", "_").replace("/", "-")
MODEL_PATH_S1=S1_WEIGHTS_PATH
MODEL_PATH_S2=S2_WEIGHTS_PATH


print("Dataset    :", DATASET)
print("Model      :", MODEL_NAME)
print("Save tag   :", MODEL_PATH)

In [ ]:
class DATASET_CLASS(Dataset):
    def __init__(self, root_dir, augment=False,
                 tile_size=TILESIZE, stride=STRIDE,
                 min_change_pixels=10,
                 neg_ratio=1.5):          
        self.root_dir  = root_dir
        self.augment   = augment
        self.tile_size = tile_size
        self.stride    = stride

        self.A = sorted(os.listdir(os.path.join(root_dir, DIRMAP["A"])))
        self.B = sorted(os.listdir(os.path.join(root_dir, DIRMAP["B"])))
        self.Y = sorted(os.listdir(os.path.join(root_dir, DIRMAP["label"])))

        probe = Image.open(os.path.join(root_dir, DIRMAP["A"], self.A[0]))
        full_W, full_H = probe.size
        probe.close()

        raw_patches = []
        if full_H == tile_size and full_W == tile_size:
            for i in range(len(self.A)):
                raw_patches.append((i, 0, 0))
        else:
            for i in range(len(self.A)):
                for r in range(0, full_H - tile_size + 1, stride):
                    for c in range(0, full_W - tile_size + 1, stride):
                        raw_patches.append((i, r, c))

        split_name = os.path.basename(root_dir)

        # neg_ratio=None for val/test: keep ALL patches, no filtering 
        if neg_ratio is None:
            self.patches = raw_patches
            print(f"  [{split_name}] {len(self.A)} images → "
                  f"{len(self.patches)} patches (NO filtering — full distribution)")
        else:
            # Train
            print(f"  [{split_name}] Scanning {len(raw_patches)} patches...", end=" ")

            change_patches    = []
            no_change_patches = []

            mask_cache_idx = -1
            mask_cache_arr = None

            for (img_idx, r, c) in raw_patches:
                if img_idx != mask_cache_idx:
                    mask_path = os.path.join(root_dir, "label", self.Y[img_idx])
                    mask_cache_arr = np.array(Image.open(mask_path).convert("L"))
                    mask_cache_idx = img_idx

                tile_mask = mask_cache_arr[r:r + tile_size, c:c + tile_size]
                n_change  = (tile_mask > 127).sum()

                if n_change >= min_change_pixels:
                    change_patches.append((img_idx, r, c))
                else:
                    no_change_patches.append((img_idx, r, c))

            # seed
            np.random.seed(42)
            np.random.shuffle(no_change_patches)

            
            max_neg  = int(len(change_patches) * neg_ratio)
            kept_neg = no_change_patches[:max_neg]

            self.patches = change_patches + kept_neg
            np.random.seed(42)
            np.random.shuffle(self.patches)

            print(f"done")
            print(f"  [{split_name}] Change patches : {len(change_patches)}")
            print(f"  [{split_name}] No-change kept : {len(kept_neg)} "
                  f"(ratio={neg_ratio}× → max {max_neg})")
            print(f"  [{split_name}] Total used     : {len(self.patches)} "
                  f"/ {len(raw_patches)} raw\n")

        self.to_tensor_img  = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std =[0.229, 0.224, 0.225])
        ])
        self.to_tensor_mask = transforms.ToTensor()

    def __len__(self):
        return len(self.patches)

    def __getitem__(self, idx):
        img_idx, r, c = self.patches[idx]
        box = (c, r, c + self.tile_size, r + self.tile_size)

        img1 = Image.open(os.path.join(self.root_dir, DIRMAP["A"], self.A[img_idx])).convert("RGB").crop(box)
        img2 = Image.open(os.path.join(self.root_dir, DIRMAP["B"], self.B[img_idx])).convert("RGB").crop(box)
        mask = Image.open(os.path.join(self.root_dir, DIRMAP["label"], self.Y[img_idx])).convert("L").crop(box)
        if self.augment:
            if torch.rand(1) < 0.5:
                img1 = TF.hflip(img1); img2 = TF.hflip(img2); mask = TF.hflip(mask)
            if torch.rand(1) < 0.5:
                img1 = TF.vflip(img1); img2 = TF.vflip(img2); mask = TF.vflip(mask)
            if torch.rand(1) < 0.3:
                angle = transforms.RandomRotation.get_params([-30, 30])
                img1 = TF.rotate(img1, angle, interpolation=InterpolationMode.BILINEAR)
                img2 = TF.rotate(img2, angle, interpolation=InterpolationMode.BILINEAR)
                mask = TF.rotate(mask, angle, interpolation=InterpolationMode.NEAREST)
            if torch.rand(1) < 0.4:
                img1 = TF.adjust_brightness(img1, torch.empty(1).uniform_(0.75, 1.25).item())
                img1 = TF.adjust_contrast(img1,   torch.empty(1).uniform_(0.75, 1.25).item())
                img2 = TF.adjust_brightness(img2, torch.empty(1).uniform_(0.75, 1.25).item())
                img2 = TF.adjust_contrast(img2,   torch.empty(1).uniform_(0.75, 1.25).item())

            #  T1/T2 random swap 
            if torch.rand(1) < 0.5:
               img1, img2 = img2, img1     

            #  HSV jitter
            if torch.rand(1) < 0.3:
                img1 = TF.adjust_hue(img1, torch.empty(1).uniform_(-0.1, 0.1).item())
                img2 = TF.adjust_hue(img2, torch.empty(1).uniform_(-0.1, 0.1).item())
                img1 = TF.adjust_saturation(img1, torch.empty(1).uniform_(0.8, 1.2).item())
                img2 = TF.adjust_saturation(img2, torch.empty(1).uniform_(0.8, 1.2).item())

        img1 = self.to_tensor_img(img1)
        img2 = self.to_tensor_img(img2)
        mask = self.to_tensor_mask(mask)
        mask = (mask > 0.5).float()

        if self.augment and torch.rand(1) < 0.2:
           img1 = img1 + torch.randn_like(img1) * 0.02
           img2 = img2 + torch.randn_like(img2) * 0.02



        return img1, img2, mask

In [ ]:
print("Building patch datasets...")
print("Using ROOT:", ROOT)

train_dataset = DATASET_CLASS(
    os.path.join(ROOT, "train"), augment=True,
    min_change_pixels=MIN_CHANGE,
    neg_ratio=1.5         
)
val_dataset = DATASET_CLASS(
    os.path.join(ROOT, "val"), augment=False,
    neg_ratio=None         
)
test_dataset = DATASET_CLASS(
    os.path.join(ROOT, "test"), augment=False,
    neg_ratio=None          
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=4, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True)

print(f"Train batches/epoch : {len(train_loader)}")
print(f"Val   batches/epoch : {len(val_loader)}")
print(f"Test  batches       : {len(test_loader)}")

In [ ]:

#Squeeze-and-Excitation
class SEBlock(nn.Module):
    def __init__(self, ch, reduction=16):
        super().__init__()
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(ch, ch // reduction, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(ch // reduction, ch, 1, bias=False),
            nn.Sigmoid()
        )
    def forward(self, x):
        return x * self.se(x)

#Decoder block with SE attention
class DecoderBlock(nn.Module):
    def __init__(self, in_ch, out_ch, skip_ch):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, out_ch, kernel_size=2, stride=2)
        self.conv = nn.Sequential(
            nn.Conv2d(out_ch + skip_ch, out_ch, kernel_size=3,
                      padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )
        self.se = SEBlock(out_ch)   

    def forward(self, x, skip):
        x = self.up(x)
        x = torch.cat([x, skip], dim=1)   
        return self.se(self.conv(x))


class DiffGate(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.gate = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(ch, ch // 8, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(ch // 8, ch, 1, bias=True),
            nn.Sigmoid()
        )
        
        nn.init.constant_(self.gate[3].bias, 3.0)
    def forward(self, f1, f2):
        g = self.gate(torch.abs(f1 - f2))  
        return f1 * g, f2 * g              

In [ ]:
class Stage1_Model_Class(nn.Module):
    def __init__(self):
        super().__init__()
        backbone = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)  # ← ResNet-50

        
        self.stage0 = nn.Sequential(backbone.conv1, backbone.bn1, backbone.relu)  
        self.stage1 = nn.Sequential(backbone.maxpool, backbone.layer1)           
        self.stage2 = backbone.layer2                                             
        self.stage3 = backbone.layer3                                            
        self.stage4 = backbone.layer4                                             

       
        self.proj4 = nn.Conv2d(2048, 512, 1, bias=False) 
        self.proj3 = nn.Conv2d(1024, 256, 1, bias=False)  

        self.diff_gate2  = DiffGate(512)
        self.diff_gate3  = DiffGate(1024)

        # Cross-attention 
        self.cross_attn4 = nn.MultiheadAttention(512, 8, batch_first=True, dropout=0.1)
        self.cross_attn3 = nn.MultiheadAttention(256, 8, batch_first=True, dropout=0.1)

        
        self.dec4 = DecoderBlock(512, 256, skip_ch=256*3) 
        self.dec3 = DecoderBlock(256, 128, skip_ch=512*3)   
        self.dec2 = DecoderBlock(128,  64, skip_ch=256*3)   
        self.dec1 = DecoderBlock( 64,  64, skip_ch= 64*3)  

        self.head = nn.Sequential(
            nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2),
            nn.ReLU(inplace=True),
            nn.Dropout2d(p=DROPOUT_P),
            nn.Conv2d(32, 1, kernel_size=1)
        )

        
        self.aux_dec4 = nn.Conv2d(256, 1, 1)
        self.aux_dec3 = nn.Conv2d(128, 1, 1)

   

    # Returns s0 s1 s2 s3 only — s4 computed after channel exchange
    def encode(self, x):
        s0 = self.stage0(x)
        s1 = self.stage1(s0)
        s2 = self.stage2(s1)
        s3 = self.stage3(s2)
        return s0, s1, s2, s3

    def _cross_attn(self, attn_module, f1_feat, f2_feat):
        B, C, H, W = f1_feat.shape
        q = f1_feat.flatten(2).permute(0, 2, 1)
        k = f2_feat.flatten(2).permute(0, 2, 1)
        out, _ = attn_module(q, k, k)
        return out.permute(0, 2, 1).reshape(B, C, H, W)

    def _rich_skip(self, a, b):
        return torch.cat([a, b, torch.abs(a - b)], dim=1)

    # Half-exchange channels
    def _channel_exchange(self, f1, f2):
        B, C, H, W = f1.shape
        mask = torch.zeros(C, device=f1.device)
        mask[1::2] = 1.0                          # odd channels exchanged
        mask = mask.view(1, C, 1, 1)
        f1_out = f1 * (1 - mask) + f2 * mask
        f2_out = f2 * (1 - mask) + f1 * mask
        return f1_out, f2_out

    def forward(self, x1, x2):
        # Encode up to stage3 only
        f1 = list(self.encode(x1))   # [s0, s1, s2, s3]
        f2 = list(self.encode(x2))
    
        # Channel exchange at stage3 
        f1[3], f2[3] = self._channel_exchange(f1[3], f2[3])
    
        # Re-run stage4 on exchanged features
        s4_1 = self.stage4(f1[3])
        s4_2 = self.stage4(f2[3])
        f1.append(s4_1)    # f1 = [s0,s1,s2,s3_exchanged,s4]
        f2.append(s4_2)
    
        # DiffGate at stage2 and stage3
        gf1_2, gf2_2 = self.diff_gate2(f1[2], f2[2])
        gf1_3, gf2_3 = self.diff_gate3(f1[3], f2[3])
    
        # Project and cross-attend at stage4 and stage3
        p1_4 = self.proj4(f1[4]); p2_4 = self.proj4(f2[4])
        p1_3 = self.proj3(gf1_3); p2_3 = self.proj3(gf2_3)
    
        attn4 = self._cross_attn(self.cross_attn4, p1_4, p2_4)
        diff4 = torch.abs(p1_4 - attn4)
    
        attn3 = self._cross_attn(self.cross_attn3, p1_3, p2_3)
        diff3 = torch.cat([p1_3, attn3, torch.abs(p1_3 - attn3)], dim=1)
    
        # Decoder 
        d4 = self.dec4(diff4, diff3)
        d3 = self.dec3(d4, self._rich_skip(gf1_2, gf2_2))    
        d2 = self.dec2(d3, self._rich_skip(f1[1], f2[1]))
        d1 = self.dec1(d2, self._rich_skip(f1[0], f2[0]))
    
        p1   = self.head(d1)
        aux4 = self.aux_dec4(d4)
        aux3 = self.aux_dec3(d3)
        return p1, aux4, aux3

loss fn

In [ ]:

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, pred, target):
        bce  = F.binary_cross_entropy_with_logits(pred, target, reduction='none')
        pt   = torch.exp(-bce)
        loss = self.alpha * (1 - pt) ** self.gamma * bce
        return loss.mean()


bce   = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([POSWEIGHT]).to(device))
focal = FocalLoss(alpha=FOCAL_ALPHA, gamma=2.0)

def dice_loss(pred, target, smooth=1e-6):
    pred   = torch.sigmoid(pred)
    pred   = pred.view(-1)
    target = target.view(-1)
    intersection = (pred * target).sum()
    return 1 - (2 * intersection + smooth) / (pred.sum() + target.sum() + smooth)

    # Focal: handles hard edge pixels, BCE: handles class balance, Dice: handles region overlap
def criterion(pred, target):
    return bce(pred, target) + dice_loss(pred, target) + 0.5 * focal(pred, target)

In [ ]:
torch.manual_seed(42)
np.random.seed(42)
model_s1 = Stage1_Model_Class().to(device)
model_s1 = Stage1_Model_Class().to(device)
if RUN_S1_TRAIN:  
    _a = torch.randn(2, 3, 256, 256).to(device)
    _b = torch.randn(2, 3, 256, 256).to(device)
    
    _out, _aux4, _aux3 = model_s1(_a, _b)  
    
    print(f"Main output shape : {_out.shape}")   
    print(f"Aux4 output shape : {_aux4.shape}")  
    print(f"Aux3 output shape : {_aux3.shape}")   
    print(f"Output max        : {_out.max().item():.4f}")
    total_params = sum(p.numel() for p in model_s1.parameters() if p.requires_grad)
    print(f"Trainable params  : {total_params/1e6:.2f}M")  

In [ ]:
if RUN_S1_TRAIN:
    # Backbone param groups for freeze/unfreeze control
    backbone_params = (list(model_s1.stage0.parameters()) +
                       list(model_s1.stage1.parameters()) +
                       list(model_s1.stage2.parameters()) +
                       list(model_s1.stage3.parameters()) +
                       list(model_s1.stage4.parameters()))
    backbone_param_ids = {id(p) for p in backbone_params}
    
    head_params = [p for p in model_s1.parameters()
                   if id(p) not in backbone_param_ids]
    
    # Freeze backbone for first N epochs
    for p in backbone_params:
        p.requires_grad = False
    
    optimizer = torch.optim.AdamW([
        {'params': backbone_params, 'lr': 1e-5},  
        {'params': head_params,     'lr': 1e-4},
    ], weight_decay=1e-4)
    
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=15, T_mult=2, eta_min=1e-6
    )

In [ ]:
def compute_metrics(pred, target):
    pred   = torch.sigmoid(pred)
    pred   = (pred > 0.5).float()
    tp = (pred  *  target      ).sum().float()
    tn = ((1-pred) * (1-target)).sum().float()
    fp = (pred  * (1-target)   ).sum().float()
    fn = ((1-pred) *  target   ).sum().float()
    return tp, tn, fp, fn

def finalize_metrics(tp, tn, fp, fn):
    eps  = 1e-8
    acc  = (tp + tn) / (tp + tn + fp + fn + eps)
    prec = tp / (tp + fp + eps)
    rec  = tp / (tp + fn + eps)
    f1   = 2 * prec * rec / (prec + rec + eps)
    iou  = tp / (tp + fp + fn + eps)  
    return {
        "acc" : acc.item(),
        "prec": prec.item(),
        "rec" : rec.item(),
        "f1"  : f1.item(),
        "iou" : iou.item()
    }

In [ ]:
if RUN_S1_TRAIN:

    history = {
        "train_loss": [], "val_loss" : [],
        "train_acc" : [], "val_acc"  : [],
        "train_prec": [], "val_prec" : [],
        "train_rec" : [], "val_rec"  : [],
        "train_f1"  : [], "val_f1"   : [],
        "train_iou" : [], "val_iou"  : [],
        'bb_lr':      [],              
        'hd_lr':      [],      
    }
    
    best_val_f1      = 0.0
    no_improve_count = 0
    best_epoch       = 0
    training_start   = time.time()
    
    print("=" * 155)
    print("  STAGE 1 TRAINING   "
          f"| {NUM_EPOCHS_S1} epochs | Early-stop patience={EARLY_STOP_PATIENCE}")
    print("=" * 155)
    print(
        f"{'Ep':>7} | {'Time':>6} | "
        f"{'TrLoss':>7} | {'TrAcc':>6} | {'TrPrec':>7} | {'TrRec':>6} | {'TrF1':>6} | {'TrIoU':>7} | "
        f"{'VlLoss':>7} | {'VlAcc':>6} | {'VlPrec':>7} | {'VlRec':>6} | {'VlF1':>6} | {'VlIoU':>7} | "
        f"{'BB_LR':>9} | {'HD_LR':>9}"
    )
    print("─" * 155)
    
    for epoch in range(NUM_EPOCHS_S1):
        start_time = time.time()
    
        # Backbone unfreeze
        if epoch == UNFREEZE_EPOCH:
            for p in backbone_params:
                p.requires_grad = True
            optimizer.param_groups[0]['lr'] = 1e-7
            optimizer.param_groups[1]['lr'] = 1e-4
            print(f"\n  → Epoch {epoch+1}: Backbone unfrozen — LR warmup begins\n")
    
            # Backbone LR warmup (epochs UNFREEZE+1 to UNFREEZE+3)
        if UNFREEZE_EPOCH < epoch <= UNFREEZE_EPOCH + 3:
            warmup_lr = 1e-7 + (1e-5 - 1e-7) * (epoch - UNFREEZE_EPOCH) / 3
            optimizer.param_groups[0]['lr'] = warmup_lr
    
    
        # Training
        model_s1.train()
        train_loss = 0.0
        tr_tp = tr_tn = tr_fp = tr_fn = 0
    
        for img1, img2, mask in tqdm(train_loader, leave=False,
                                      disable=not SHOW_PROGRESS,
                                      desc=f"Ep {epoch+1:3d} train"):
            img1, img2, mask = img1.to(device), img2.to(device), mask.to(device)
            optimizer.zero_grad()
    
            with torch.amp.autocast('cuda'):
                pred, aux4, aux3 = model_s1(img1, img2)
                mask_16 = F.interpolate(mask, size=aux4.shape[-2:], mode='nearest')
                mask_32 = F.interpolate(mask, size=aux3.shape[-2:], mode='nearest')
                loss = (criterion(pred, mask)
                        + 0.4 * criterion(aux4, mask_16)
                        + 0.2 * criterion(aux3, mask_32))
    
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model_s1.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
    
            train_loss += loss.item()
            tp, tn, fp, fn = compute_metrics(pred.detach(), mask)
            tr_tp += tp; tr_tn += tn; tr_fp += fp; tr_fn += fn
    
        # Validation
        model_s1.eval()
        val_loss = 0.0
        v_tp = v_tn = v_fp = v_fn = 0
    
        with torch.no_grad():
            for img1, img2, mask in val_loader:
                img1, img2, mask = img1.to(device), img2.to(device), mask.to(device)
                with torch.amp.autocast('cuda'):
                    pred, _, _ = model_s1(img1, img2)
                val_loss += criterion(pred, mask).item()
                tp, tn, fp, fn = compute_metrics(pred, mask)
                v_tp += tp; v_tn += tn; v_fp += fp; v_fn += fn
    

        tr_m  = finalize_metrics(tr_tp, tr_tn, tr_fp, tr_fn)
        val_m = finalize_metrics(v_tp,  v_tn,  v_fp,  v_fn)
    
        scheduler.step()
        current_lr_bb   = scheduler.get_last_lr()[0]
        current_lr_head = scheduler.get_last_lr()[1]
    
        history['bb_lr'].append(current_lr_bb)    
        history['hd_lr'].append(current_lr_head)  
        history["train_loss"].append(train_loss / len(train_loader))
        history["val_loss"  ].append(val_loss   / len(val_loader))
        history["train_acc" ].append(tr_m["acc"]);  history["val_acc" ].append(val_m["acc"])
        history["train_prec"].append(tr_m["prec"]); history["val_prec"].append(val_m["prec"])
        history["train_rec" ].append(tr_m["rec"]);  history["val_rec" ].append(val_m["rec"])
        history["train_f1"  ].append(tr_m["f1"]);   history["val_f1"  ].append(val_m["f1"])
        history["train_iou" ].append(tr_m["iou"]);  history["val_iou" ].append(val_m["iou"])
    
        # saving best model and early stop counting
        if val_m["f1"] > best_val_f1:
            best_val_f1      = val_m["f1"]
            best_epoch       = epoch + 1
            no_improve_count = 0
            MODEL_PATH_S1=f"{MODEL_PATH}_stage1.pth"
            torch.save(model_s1.state_dict(),MODEL_PATH_S1 )
            saved_marker = "BEST"
        else:
            no_improve_count += 1
            saved_marker = f"(no improvement {no_improve_count}/{EARLY_STOP_PATIENCE})"
    
        # checkpoint
        if (epoch + 1) % CHECKPOINT_EVERY == 0:
            ckpt_path = f"ckpt_stage1_ep{epoch+1}.pth"
            torch.save(model_s1.state_dict(), ckpt_path)
            saved_marker += f"  [ckpt saved]"
    
        # displaying epoch metrics
        m, s = divmod(int(time.time() - start_time), 60)
        ep_str = f"{epoch+1}/{NUM_EPOCHS_S1}"  
        print(
            f"{ep_str:7} | {m:2d}m {s:2d}s | "
            f"{history['train_loss'][-1]:7.4f} | "
            f"{tr_m['acc']:6.4f} | {tr_m['prec']:7.4f} | {tr_m['rec']:6.4f} | "
            f"{tr_m['f1']:6.4f} | {tr_m['iou']:7.4f} | "
            f"{history['val_loss'][-1]:7.4f} | "
            f"{val_m['acc']:6.4f} | {val_m['prec']:7.4f} | {val_m['rec']:6.4f} | "
            f"{val_m['f1']:6.4f} | {val_m['iou']:7.4f} | "
            f"{current_lr_bb:.3e} | {current_lr_head:.3e}"
            f"{saved_marker}"
        )
    
        if (epoch + 1) % 10 == 0:
            elapsed = int(time.time() - training_start)
            em, es  = divmod(elapsed, 60)
            print("─" * 155)
            print(f" Best so far: Val F1={best_val_f1:.4f} @ Ep {best_epoch}"
                  f"  |  Elapsed: {em}m {es}s"
                  f"  |  No-improve streak: {no_improve_count}/{EARLY_STOP_PATIENCE}")
            print("─" * 155)
    
        # Early stopping 
        if no_improve_count >= EARLY_STOP_PATIENCE:
            print(f"\n  Early stopping at epoch {epoch+1} "
                  f"— no Val F1 improvement for {EARLY_STOP_PATIENCE} epochs")
            break
    
    # Final training summary
    total_time = int(time.time() - training_start)
    tm, ts = divmod(total_time, 60)
    th, tm = divmod(tm, 60)
    print("\n" + "=" * 155)
    print(f"  STAGE 1 COMPLETE")
    print(f"  Best Val F1 : {best_val_f1:.4f}  @ Epoch {best_epoch}")
    print(f"  Total Time  : {th}h {tm}m {ts}s")
    print(f"  Saved       :{MODEL_PATH_S1}")
    print("=" * 155)


In [ ]:
if RUN_S1_TRAIN:

    epochs = list(range(1, len(history['train_f1']) + 1))

    BG      = 'white'
    AX_BG   = '#f7f7f7'
    GRID_C  = '#dddddd'
    TEXT_C  = '#222222'
    SPINE_C = '#bbbbbb'

    def _style_ax(ax):
        ax.set_facecolor(AX_BG)
        ax.tick_params(colors=TEXT_C, labelsize=9)
        ax.xaxis.label.set_color(TEXT_C)
        ax.yaxis.label.set_color(TEXT_C)
        ax.title.set_color(TEXT_C)
        ax.title.set_fontweight('bold')
        for sp in ax.spines.values():
            sp.set_edgecolor(SPINE_C)
        ax.grid(False)
        ax.xaxis.grid(True, color=GRID_C, linewidth=0.7, linestyle='-', alpha=0.8)
        ax.yaxis.grid(False)
        ax.set_axisbelow(True)

    def _best_vline(ax, ep, label_text):
        ax.axvline(x=ep, color='#16a34a', linestyle=':', linewidth=2)
        ylim = ax.get_ylim()
        y_pos = ylim[0] + 0.65 * (ylim[1] - ylim[0])
        ax.text(ep + 0.3, y_pos, label_text,
                color='#16a34a', fontsize=7.5, fontweight='bold',
                va='bottom', rotation=90, clip_on=True)

    def _save_show(fig, fname, right=0.95):
        fig.subplots_adjust(left=0.11, right=right, top=0.88, bottom=0.13)
        plt.savefig(fname, dpi=140, bbox_inches='tight',
                    facecolor=BG, pad_inches=0.3)
        plt.show()
        plt.close(fig)
        print(f'Saved: {fname}')

    best_f1 = max(history['val_f1'])
    best_ep = history['val_f1'].index(best_f1) + 1

    #  Plot1: Train vs Val Loss 
    fig, ax = plt.subplots(figsize=(10, 5))
    fig.patch.set_facecolor(BG)
    _style_ax(ax)
    ax.plot(epochs, history['train_loss'], label='Train Loss',
            color='#2563eb', linewidth=2)
    ax.plot(epochs, history['val_loss'],   label='Val Loss',
            color='#dc2626', linewidth=2)
    ax.set_title('Train vs Val Loss', fontsize=13)
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
    ax.legend(fontsize=10, framealpha=0.9, loc='upper right')
    ax.set_ylim(bottom=0)
    _save_show(fig, 's1_a1_loss.png')

    #  Plot2: Train vs Val F1 
    fig, ax = plt.subplots(figsize=(10, 5))
    fig.patch.set_facecolor(BG)
    _style_ax(ax)
    ax.plot(epochs, history['train_f1'], label='Train F1',
            color='#2563eb', linewidth=2)
    ax.plot(epochs, history['val_f1'],   label='Val F1',
            color='#dc2626', linewidth=2)
    _best_vline(ax, best_ep, 'Best F1 Epoch')
    ax.set_title(' F1 Score', fontsize=13)
    ax.set_xlabel('Epoch'); ax.set_ylabel('F1')
    best_handle = Line2D([0], [0], color='#16a34a', linestyle=':', linewidth=2,
                         label=f'Best Val F1={best_f1:.4f} (Ep {best_ep})')
    h, l = ax.get_legend_handles_labels()
    ax.legend(handles=h + [best_handle], fontsize=9,
              framealpha=0.9, loc='lower right')
    _save_show(fig, 's1_a2_f1.png')

    #  Plot3: Train vs Val IoU 
    fig, ax = plt.subplots(figsize=(10, 5))
    fig.patch.set_facecolor(BG)
    _style_ax(ax)
    ax.plot(epochs, history['train_iou'], label='Train IoU',
            color='#0891b2', linewidth=2)
    ax.plot(epochs, history['val_iou'],   label='Val IoU',
            color='#e11d48', linewidth=2)
    ax.set_title('IoU Score', fontsize=13)
    ax.set_xlabel('Epoch'); ax.set_ylabel('IoU')
    ax.legend(fontsize=10, framealpha=0.9, loc='lower right')
    _save_show(fig, 's1_a3_iou.png')

    #  Plot4: Train vs Val Precision 
    fig, ax = plt.subplots(figsize=(10, 5))
    fig.patch.set_facecolor(BG)
    _style_ax(ax)
    ax.plot(epochs, history['train_prec'], label='Train Precision',
            color='#2563eb', linewidth=2)
    ax.plot(epochs, history['val_prec'],   label='Val Precision',
            color='#dc2626', linewidth=2)
    ax.set_title('Precision', fontsize=13)
    ax.set_xlabel('Epoch'); ax.set_ylabel('Precision')
    ax.legend(fontsize=10, framealpha=0.9, loc='lower right')
    _save_show(fig, 's1_a4_precision.png')

    #  Plot5: Train vs Val Recall 
    fig, ax = plt.subplots(figsize=(10, 5))
    fig.patch.set_facecolor(BG)
    _style_ax(ax)
    ax.plot(epochs, history['train_rec'], label='Train Recall',
            color='#0891b2', linewidth=2)
    ax.plot(epochs, history['val_rec'],   label='Val Recall',
            color='#e11d48', linewidth=2)
    ax.set_title('A5 · Recall', fontsize=13)
    ax.set_xlabel('Epoch'); ax.set_ylabel('Recall')
    ax.legend(fontsize=10, framealpha=0.9, loc='lower right')
    _save_show(fig, 's1_a5_recall.png')

In [ ]:
# Stage 1 Test
if EVAL_S1:
    s1_ckpt = MODEL_PATH_S1
    model_s1.load_state_dict(torch.load(s1_ckpt, map_location=device))
    model_s1.eval()
    
    def predict_tta(model_s1, img1, img2):
     
        preds = []
        flip_pairs = [
            (lambda x: x,          lambda x: x),           # original
            (TF.hflip,              TF.hflip),              # hflip → unflip pred
            (TF.vflip,              TF.vflip),              # vflip → unflip pred
            (lambda x: TF.hflip(TF.vflip(x)),
             lambda x: TF.hflip(TF.vflip(x))),             # both flips
        ]
    
    
    
        
        with torch.no_grad():
            for flip_in, flip_out in flip_pairs:
                i1 = torch.stack([flip_in(img1[b]) for b in range(img1.shape[0])])
                i2 = torch.stack([flip_in(img2[b]) for b in range(img2.shape[0])])
                with torch.amp.autocast('cuda'):
                    p, _, _ = model_s1(i1, i2)
                p_sig = torch.sigmoid(p)
                p_unflipped = torch.stack([flip_out(p_sig[b]) for b in range(p_sig.shape[0])])
                preds.append(p_unflipped)
        return torch.stack(preds).mean(0)  
    
    test_loss = 0.0
    t_tp = t_tn = t_fp = t_fn = 0

    total_test_start = time.time()

    total_img_pairs = len(test_loader.dataset)
    
    with torch.no_grad():
        for img1, img2, mask in test_loader:
            img1, img2, mask = img1.to(device), img2.to(device), mask.to(device)
            avg_prob = predict_tta(model_s1, img1, img2)          
            pred_bin = (avg_prob > 0.5).float()
            tp = (pred_bin * mask).sum().float()
            tn = ((1-pred_bin)*(1-mask)).sum().float()
            fp = (pred_bin*(1-mask)).sum().float()
            fn = ((1-pred_bin)*mask).sum().float()
            t_tp += tp; t_tn += tn; t_fp += fp; t_fn += fn


    total_test_time = time.time() - total_test_start
    time_per_img_pair = total_test_time / total_img_pairs
    
    testm = finalize_metrics(t_tp, t_tn, t_fp, t_fn)
    s1_test = testm
    print("\n" + "═" * 50)
    print("  STAGE 1 Test Metrics")
    print("═" * 50)

    print(f"  Dataset              : {DATASET}")
    print(f"  Total image pairs    : {total_img_pairs}")
    print(f"  Total test time      : {total_test_time:.2f} sec")
    print(f"  Time / image pair    : {time_per_img_pair:.4f} sec")
    
    print("=" * 56)
    print(f"  Accuracy  : {testm['acc']:.4f}")
    print(f"  Precision : {testm['prec']:.4f}")
    print(f"  Recall    : {testm['rec']:.4f}")
    print(f"  F1 Score  : {testm['f1']:.4f}")
    print(f"  IoU       : {testm['iou']:.4f}") 

    print("═" * 50)

In [ ]:
if EVAL_S1:  
      
    fig, ax = plt.subplots(1, 1, figsize=(7, 5))
    fig.suptitle('Stage 1 Test Results', fontsize=13, fontweight='bold')
    fig.patch.set_facecolor('#0d0d1a')
    
    ax.set_facecolor('#12122a')
    ax.tick_params(colors='#aaaacc')
    ax.xaxis.label.set_color('#aaaacc')
    ax.yaxis.label.set_color('#aaaacc')
    ax.title.set_color('white')
    for sp in ax.spines.values():
        sp.set_edgecolor('#2a2a4a')
    
    # ── Plot1: Our test metrics
    metric_names = ['Accuracy', 'Precision', 'Recall', 'F1', 'IoU']
    metric_vals  = [testm['acc'], testm['prec'], testm['rec'], testm['f1'], testm['iou']]
    bar_colors   = ['#5b8cff', '#ff6b6b', '#ffa94d', '#4dff91', '#cc88ff']
    
    bars = ax.bar(metric_names, metric_vals, color=bar_colors, width=0.55)
    
    for bar, val in zip(bars, metric_vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                f'{val:.4f}', ha='center', va='bottom',
                color='white', fontsize=10, fontweight='bold')
    
    ax.set_ylim(min(metric_vals) - 0.05, 1.0)
    ax.set_title(' Stage 1 Test Metrics')
    ax.set_ylabel('Score')
    
    plt.tight_layout()
    plt.savefig('s1_test_metrics.png', dpi=140, bbox_inches='tight', facecolor='#0d0d1a')
    plt.show()
    
    print("Saved: s1_test_metrics.png")

**Stage 2**

In [ ]:
#S2 dataset- config 
S2_CONFIG = {
    'LEVIR-CD': {
        'aspp_dilations':       (1, 3, 6),   
        'boundary_kernel':      5,
        'lr':                   2e-4,
        'delta_scale_init':     0.5,
    },
    'LIM-CD': {
        'aspp_dilations':       (1, 3, 6),
        'boundary_kernel':      7,
        'lr':                   1e-4,
        'delta_scale_init':     0.5,
    },
}
s2_cfg = S2_CONFIG[DATASET]
print(f"S2 config for {DATASET}: {s2_cfg}")

In [ ]:
# Stage 2 Model: Same as Stage 1 but with forward_with_stage2_features() for feature extraction

def _forward_s2(self, x1, x2):

    # Encode up to stage3 
    f1 = list(self.encode(x1))
    f2 = list(self.encode(x2))

    # Channel exchange at stage3 
    f1[3], f2[3] = self._channel_exchange(f1[3], f2[3])

    # stage4 on exchanged features
    s4_1 = self.stage4(f1[3]);  s4_2 = self.stage4(f2[3])
    f1.append(s4_1);            f2.append(s4_2)

    # ── DiffGate ── 
    gf1_2, gf2_2 = self.diff_gate2(f1[2], f2[2])
    gf1_3, gf2_3 = self.diff_gate3(f1[3], f2[3])

    # Project + cross-attend
    p1_4 = self.proj4(f1[4]);  p2_4 = self.proj4(f2[4])
    p1_3 = self.proj3(gf1_3);  p2_3 = self.proj3(gf2_3)

    attn4      = self._cross_attn(self.cross_attn4, p1_4, p2_4)
    diff4      = torch.abs(p1_4 - attn4)

    attn3      = self._cross_attn(self.cross_attn3, p1_3, p2_3)
    diff3_attn = torch.abs(p1_3 - attn3)
    diff3_dec  = torch.cat([p1_3, attn3, diff3_attn], dim=1)

    # Decoder 
    d4 = self.dec4(diff4, diff3_dec)
    d3 = self.dec3(d4,   self._rich_skip(gf1_2, gf2_2))
    d2 = self.dec2(d3,   self._rich_skip(f1[1], f2[1]))
    d1 = self.dec1(d2,   self._rich_skip(f1[0], f2[0]))
    p1 = self.head(d1)

    diff2_raw = torch.abs(f1[2] - f2[2])   
    return p1, diff4, diff3_attn, diff2_raw

Stage1_Model_Class.forward_with_stage2_features = _forward_s2


# ── Load & freeze Stage1 ──
model_s1 = Stage1_Model_Class().to(device)
s1_ckpt = MODEL_PATH_S1  
model_s1.load_state_dict(torch.load(s1_ckpt, map_location=device))
model_s1.eval()
for p in model_s1.parameters():
    p.requires_grad = False

print("=" * 54)
print("  STAGE 1 LOADED & FROZEN")
print("=" * 54)
s1_params = sum(p.numel() for p in model_s1.parameters())
print(f"  Params (frozen): {s1_params/1e6:.2f}M")

# Quick sanity check of Stage2 interface
with torch.no_grad():
    _a = torch.randn(2, 3, 256, 256).to(device)
    _b = torch.randn(2, 3, 256, 256).to(device)
    _p1, _d4, _d3, _d2 = model_s1.forward_with_stage2_features(_a, _b)

print(f"  p1 shape        : {_p1.shape}")     
print(f"  diff4 shape     : {_d4.shape}")       
print(f"  diff3_attn shape: {_d3.shape}")       
print(f"  diff2_raw shape : {_d2.shape}")      


flip_pairs = [
        (lambda x: x,          lambda x: x),           
        (TF.hflip,              TF.hflip),             
        (TF.vflip,              TF.vflip),              
        (lambda x: TF.hflip(TF.vflip(x)),
         lambda x: TF.hflip(TF.vflip(x))),            
]

print("=" * 54)
print("  Moving to Stage 2: Object-Based Refinement")
print("=" * 54)

In [ ]:
class RegionSegmentationHead(nn.Module):
    def __init__(self):
        super().__init__()

        self.proj_diff2 = nn.Sequential(
            nn.Conv2d(512, 128, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True)
        )

        self.region_conv1 = nn.Sequential(
            nn.Conv2d(257, 128, 3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True)
        )

        self.region_conv2 = nn.Sequential(
            nn.Conv2d(256, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True)
        )

        self.region_out = nn.Conv2d(64, 1, 1)

    def forward(self, diff3_attn, diff2_raw, p1_sig):

        diff2 = self.proj_diff2(diff2_raw)

        p1_16 = F.interpolate(
            p1_sig,
            size=diff3_attn.shape[-2:],
            mode='bilinear',
            align_corners=False
        )

        x = self.region_conv1(torch.cat([diff3_attn, p1_16], dim=1))

        x = F.interpolate(
            x,
            size=diff2.shape[-2:],
            mode='bilinear',
            align_corners=False
        )

        x = self.region_conv2(torch.cat([x, diff2], dim=1))

        return self.region_out(x)

In [ ]:
class ObjectAggregationModule(nn.Module):
    def __init__(self):
        super().__init__()
        self.agg_proj = nn.Sequential(
            nn.Conv2d(512, 128, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True))
        # NEW: global context from diff4
        self.ctx_proj = nn.Sequential(
            nn.Conv2d(512, 128, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True))
        self.agg_fuse = nn.Sequential(
            nn.Conv2d(384, 128, 1, bias=False),  # 128+128+128
            nn.BatchNorm2d(128), nn.ReLU(inplace=True))

    def forward(self, diff4, diff2_raw, region_map):
        feat     = self.agg_proj(diff2_raw)                      # B,128,32,32
        ctx      = self.ctx_proj(diff4)                           # B,128,8,8
        ctx      = F.interpolate(ctx, size=feat.shape[-2:],
                                  mode='bilinear', align_corners=False)  # B,128,32,32
        region16 = F.interpolate(torch.sigmoid(region_map),
                                  size=feat.shape[-2:], mode='bilinear', align_corners=False)
        w        = region16.sum(dim=(2,3), keepdim=True) + 1e-6
        proto    = (feat * region16).sum(dim=(2,3), keepdim=True) / w
        obj_feat = self.agg_fuse(torch.cat([feat, ctx,
                                             proto.expand_as(feat)], dim=1))
        return obj_feat

In [ ]:
class ASPPRefinement(nn.Module):
    def __init__(self, dilations=(1,3,6)):
        super().__init__()

        self.proj3 = nn.Sequential(
            nn.Conv2d(128, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True)
        )
        self.up2 = nn.ConvTranspose2d(64, 64, 2, stride=2)
        self.up1 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.up0 = nn.ConvTranspose2d(32, 16, 2, stride=2)

        self.region_proj = nn.Conv2d(1, 64, 1, bias=False)

        self.fuse_region = nn.Sequential(
            nn.Conv2d(128, 64, 1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True)
        )

        d1, d2, d3 = dilations

        def branch(d):
            return nn.Sequential(
                nn.Conv2d(64, 32, 3, padding=d, dilation=d, bias=False),
                nn.BatchNorm2d(32),
                nn.ReLU(inplace=True)
            )

        self.aspp_d1 = branch(d1)
        self.aspp_d2 = branch(d2)
        self.aspp_d3 = branch(d3)

        self.aspp_gap = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(64, 32, 1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True)
        )

        self.aspp_fuse = nn.Sequential(
            nn.Conv2d(128, 64, 1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Dropout2d(0.1)
        )

    def forward(self, obj_feat, region_map):
        x = self.proj3(obj_feat)
        rg = self.region_proj(torch.sigmoid(region_map))
        x = self.fuse_region(torch.cat([x, rg], dim=1))
        x = self.up2(x)
    
        
        a1  = self.aspp_d1(x)
        a2  = self.aspp_d2(x)
        a3  = self.aspp_d3(x)
        gap = self.aspp_gap(x).expand_as(a1)   # ← now uses saved a1 for shape
        x   = self.aspp_fuse(torch.cat([a1, a2, a3, gap], dim=1))
    
        x = self.up1(x)
        x = self.up0(x)
        return x

In [ ]:
class Stage2_Model_Class(nn.Module):
    def __init__(self, aspp_dilations=(1,3,6), delta_scale_init=0.5):  # 0.5 not 0.1
        super().__init__()
        self.region_head  = RegionSegmentationHead()
        self.object_agg   = ObjectAggregationModule()
        self.refine       = ASPPRefinement(aspp_dilations)
        self.delta_head   = nn.Conv2d(16, 1, 1)
        nn.init.zeros_(self.delta_head.weight)
        nn.init.zeros_(self.delta_head.bias)
        self.delta_scale  = nn.Parameter(torch.tensor(delta_scale_init))

    def forward(self, diff3_attn, diff4, diff2_raw, p1_logit):
        p1_sig     = torch.sigmoid(p1_logit)

        # gate 
        gate       = torch.clamp(4.0 * p1_sig * (1.0 - p1_sig), min=0.2)

        region_map = self.region_head(diff3_attn, diff2_raw, p1_sig)
        obj_feat   = self.object_agg(diff4, diff2_raw, region_map)  
        refined    = self.refine(obj_feat, region_map)

        
        delta      = torch.tanh(self.delta_head(refined))* torch.clamp(self.delta_scale, 0.05, 2.0)
        p2_logit   = p1_logit + gate * delta

        return p2_logit, region_map, gate

In [ ]:
bce_s2 = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([POSWEIGHT]).to(device))
boundary_kernel = s2_cfg['boundary_kernel']
def boundary_loss(pred_logit, target, kernel_size=boundary_kernel):

    kern = torch.ones(1,1,kernel_size,kernel_size, device=target.device)/(kernel_size*kernel_size)

    dilated = F.conv2d(target, kern, padding=kernel_size//2).clamp(0,1)
    eroded  = 1 - F.conv2d(1-target, kern, padding=kernel_size//2).clamp(0,1)

    band = (dilated-eroded).clamp(0,1)

    bce = F.binary_cross_entropy_with_logits(pred_logit, target, reduction='none')

    return (bce*(1+3*band)).mean()


def criterion_s2(p2_logit, target, region_logit, p1_logit, gate):
    loss_bce  = bce_s2(p2_logit, target)
    loss_dice = dice_loss(p2_logit, target)

    # Boundary loss
    loss_bnd  = boundary_loss(p2_logit, target)

    # Region segmentation auxiliary loss
    region_tgt = F.interpolate(target, size=region_logit.shape[-2:], mode='nearest')
    loss_reg   = F.binary_cross_entropy_with_logits(region_logit, region_tgt)

    
    correction = torch.sigmoid(p2_logit) - torch.sigmoid(p1_logit).detach()
    correct_dir = (2 * target - 1) * correction       # positive if correcting in right direction
    loss_corr  = -(gate.detach() * correct_dir).mean() # maximize correct-direction corrections

    return loss_bce + loss_dice + 0.3*loss_reg + 0.5*loss_bnd + 0.3*loss_corr

In [ ]:
torch.manual_seed(42)
np.random.seed(42)

model_s2 = Stage2_Model_Class(
    aspp_dilations=s2_cfg['aspp_dilations'],
    delta_scale_init=s2_cfg['delta_scale_init']
).to(device)

with torch.no_grad():
    PATCH = TILESIZE

    _a = torch.randn(2, 3, PATCH, PATCH).to(device)
    _b = torch.randn(2, 3, PATCH, PATCH).to(device)

    _p1, _d4, _d3, _d2 = model_s1.forward_with_stage2_features(_a, _b)

    _p2, _reg, _gate = model_s2(_d3, _d4, _d2, _p1)

print(f"P1 shape     : {_p1.shape}")
print(f"P2 shape     : {_p2.shape}")
print(f"Region shape : {_reg.shape}")
print(f"Gate shape   : {_gate.shape}")
print(f"Gate range   : {_gate.min():.4f} → {_gate.max():.4f}")

s2_params = sum(p.numel() for p in model_s2.parameters() if p.requires_grad)
print(f"Stage2 trainable params : {s2_params/1e6:.2f}M")
print("Stage2 sanity check passed")

In [ ]:
if RUN_S2_TRAIN:

    for p in model_s1.parameters():
        p.requires_grad = False

    optimizer_s2 = torch.optim.AdamW(
        model_s2.parameters(),
        lr=s2_cfg['lr'],
        weight_decay=1e-4
    )

    scheduler_s2 = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer_s2,
        T_0=10,
        T_mult=2,
        eta_min=1e-6
    )

    total_s2 = sum(p.numel() for p in model_s2.parameters())
    print(f"S2 trainable: {total_s2/1e6:.3f}M")

In [ ]:
if RUN_S2_TRAIN:
    s2_history = {
        'train_loss': [], 'val_loss': [],
        'train_prec': [], 'val_prec': [],
        'train_rec':  [], 'val_rec':  [],
        'train_f1':   [], 'val_f1':   [],
        'train_iou':  [], 'val_iou':  [],
        'lr':         [],
        'gate_mean':  [],
        'mean_corr':  [],
        'max_corr':   [],
        }
    
    best_s2_f1         = 0.0
    no_improve_s2      = 0
    best_s2_epoch      = 0
    s2_training_start  = time.time()
    
    print("=" * 112)
    print("  STAGE 2 TRAINING  —  Object-Based Refinement  "
          f"| {NUM_EPOCHS_S2} epochs | Early-stop patience={EARLY_STOP_PATIENCE_S2}")
    print("=" * 112)
    print(
        f"{'Ep':>6} | {'Time':>6} | "
        f"{'TrLoss':>7} | {'TrPrec':>7} | {'TrRec':>6} | {'TrF1':>6} | {'TrIoU':>7} | "
        f"{'VlLoss':>7} | {'VlPrec':>7} | {'VlRec':>6} | {'VlF1':>6} | {'VlIoU':>7} | "
        f"{'LR':>9}"
    )
    print("─" * 112)
    
    for epoch in range(NUM_EPOCHS_S2):
        start = time.time()
        model_s2.train()
        tr_loss = 0.0
        tr_tp = tr_tn = tr_fp = tr_fn = 0
        ep_gate_sum = 0.0
        ep_mean_sum = 0.0
        ep_max_sum  = 0.0
        n_batches   = 0
    
        # Train
        for img1, img2, mask in tqdm(train_loader, leave=False,
                                      disable=not SHOW_PROGRESS,
                                      desc=f"S2 Ep {epoch+1:3d} train"):
            img1, img2, mask = img1.to(device), img2.to(device), mask.to(device)
    
            with torch.no_grad():
                with torch.amp.autocast('cuda'):
                    p1, diff4, diff3_attn, diff2_raw = model_s1.forward_with_stage2_features(img1, img2)
    
            optimizer_s2.zero_grad()
            with torch.amp.autocast('cuda'):
                p2, region, gate = model_s2(diff3_attn, diff4, diff2_raw, p1)
                loss = criterion_s2(p2, mask, region,p1,gate)
                
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer_s2)
            torch.nn.utils.clip_grad_norm_(model_s2.parameters(), 1.0)
            scaler.step(optimizer_s2)
            scaler.update()
            

            
            with torch.no_grad():
                delta = (torch.sigmoid(p2) - torch.sigmoid(p1)).abs()
            
                ep_gate_sum += gate.mean().item()
                ep_mean_sum += delta.mean().item()
                ep_max_sum  += delta.max().item()
                
            n_batches += 1
            tr_loss += loss.item()
            tp, tn, fp, fn = compute_metrics(p2.detach(), mask)
            tr_tp += tp; tr_tn += tn; tr_fp += fp; tr_fn += fn

        ep_gate = ep_gate_sum / n_batches
        ep_mean = ep_mean_sum / n_batches
        ep_max  = ep_max_sum / n_batches
    
        # Validation
        model_s2.eval()
        val_loss = 0.0
        v_tp = v_tn = v_fp = v_fn = 0
    
        with torch.no_grad():
            for img1, img2, mask in val_loader:
                img1, img2, mask = img1.to(device), img2.to(device), mask.to(device)
                with torch.amp.autocast('cuda'):
                    p1, diff4, diff3_attn, diff2_raw = model_s1.forward_with_stage2_features(img1, img2)
                    p2,region_val, _gate = model_s2(diff3_attn, diff4, diff2_raw, p1)
                val_loss += criterion_s2(p2, mask,region_val,p1,_gate).item()
                tp, tn, fp, fn = compute_metrics(p2, mask)
                v_tp += tp; v_tn += tn; v_fp += fp; v_fn += fn
    
        
        tr_m  = finalize_metrics(tr_tp, tr_tn, tr_fp, tr_fn)
        val_m = finalize_metrics(v_tp,  v_tn,  v_fp,  v_fn)
        
        lr = scheduler_s2.get_last_lr()[0]
    
        s2_history['train_loss'].append(tr_loss / len(train_loader))
        s2_history['val_loss'].append(val_loss / len(val_loader))
        s2_history['train_prec'].append(tr_m['prec'])
        s2_history['val_prec'].append(val_m['prec'])
        s2_history['train_rec'].append(tr_m['rec'])
        s2_history['val_rec'].append(val_m['rec'])
        s2_history['train_f1'].append(tr_m['f1'])
        s2_history['val_f1'].append(val_m['f1'])
        s2_history['train_iou'].append(tr_m['iou'])
        s2_history['val_iou'].append(val_m['iou'])
        s2_history['lr'].append(lr)
        
    
        # Save best model
        if val_m["f1"] > best_s2_f1:
            best_s2_f1    = val_m["f1"]
            best_s2_epoch = epoch + 1
            no_improve_s2 = 0
            MODEL_PATH_S2=f"{MODEL_PATH}_stage2.pth"
            torch.save(model_s2.state_dict(), MODEL_PATH_S2)
            marker = " BEST"
        else:
            no_improve_s2 += 1
            marker = f"  (no↑ {no_improve_s2}/{EARLY_STOP_PATIENCE_S2})"
    
        # checkpoint 
        if (epoch + 1) % CHECKPOINT_EVERY_S2 == 0:
            ckpt_path = f"ckpt_stage2_ep{epoch+1}.pth"
            torch.save(model_s2.state_dict(), ckpt_path)
            marker += "  [ckpt saved]"
    
        # Display metrics
        m, s = divmod(int(time.time() - start), 60)
        ep_str = f"{epoch+1}/{NUM_EPOCHS_S2}" 
        print(
            f"{ep_str:6} | {m:2d}m {s:2d}s | "
            f"{tr_loss/len(train_loader):7.4f} | "
            f"{tr_m['prec']:7.4f} | {tr_m['rec']:6.4f} | "
            f"{tr_m['f1']:6.4f} | {tr_m['iou']:7.4f} | "
            f"{val_loss/len(val_loader):7.4f} | "
            f"{val_m['prec']:7.4f} | {val_m['rec']:6.4f} | "
            f"{val_m['f1']:6.4f} | {val_m['iou']:7.4f} | "
            f"{lr:.3e}{marker} | Gate={ep_gate:.3f} | Δμ={ep_mean:.3f} | Δmax={ep_max:.3f}"
        )
    
        if (epoch + 1) % 10 == 0:
            elapsed = int(time.time() - s2_training_start)
            em, es  = divmod(elapsed, 60)
            print("─" * 112)
            print(f"  Best so far: Val F1={best_s2_f1:.4f} @ Ep {best_s2_epoch}"
                  f"  |  Elapsed: {em}m {es}s"
                  f"  |  No-improve streak: {no_improve_s2}/{EARLY_STOP_PATIENCE_S2}")
            print("─" * 112)
    
        # Early stopping 
        if no_improve_s2 >= EARLY_STOP_PATIENCE_S2:
            print(f"\n  Early stopping at epoch {epoch+1} "
                  f"— no Val F1 improvement for {EARLY_STOP_PATIENCE_S2} epochs")
            break

        scheduler_s2.step(epoch)
    
    # Stage 2 summary
    total_s2 = int(time.time() - s2_training_start)
    tm2, ts2  = divmod(total_s2, 60)
    th2, tm2  = divmod(tm2, 60)
    print("\n" + "=" * 112)
    print(f"  STAGE 2 COMPLETE")
    print(f"  Best Val F1 : {best_s2_f1:.4f}  @ Epoch {best_s2_epoch}")
    print(f"  Total Time  : {th2}h {tm2}m {ts2}s")
    print(f"  Saved       : {MODEL_PATH_S2}")
    print("=" * 112)


In [ ]:
if RUN_S2_TRAIN:

    s2_epochs = list(range(1, len(s2_history['val_f1']) + 1))

    BG      = 'white'
    AX_BG   = '#f7f7f7'
    GRID_C  = '#dddddd'
    TEXT_C  = '#222222'
    SPINE_C = '#bbbbbb'

    def _style_ax(ax):
        ax.set_facecolor(AX_BG)
        ax.tick_params(colors=TEXT_C, labelsize=9)
        ax.xaxis.label.set_color(TEXT_C)
        ax.yaxis.label.set_color(TEXT_C)
        ax.title.set_color(TEXT_C)
        ax.title.set_fontweight('bold')
        for sp in ax.spines.values():
            sp.set_edgecolor(SPINE_C)
        ax.grid(False)
        ax.xaxis.grid(True, color=GRID_C, linewidth=0.7, linestyle='-', alpha=0.8)
        ax.yaxis.grid(False)
        ax.set_axisbelow(True)

    def _best_vline(ax, ep, label_text):
        ax.axvline(x=ep, color='#16a34a', linestyle=':', linewidth=2)
        ylim = ax.get_ylim()
        y_pos = ylim[0] + 0.65 * (ylim[1] - ylim[0])
        ax.text(ep + 0.3, y_pos, label_text,
                color='#16a34a', fontsize=7.5, fontweight='bold',
                va='bottom', rotation=90, clip_on=True)

    def _save_show(fig, fname, right=0.95):
        fig.subplots_adjust(left=0.11, right=right, top=0.88, bottom=0.13)
        plt.savefig(fname, dpi=140, bbox_inches='tight',
                    facecolor=BG, pad_inches=0.3)
        plt.show()
        plt.close(fig)
        print(f'Saved: {fname}')

    best_s2_f1 = max(s2_history['val_f1'])
    best_s2_ep = s2_history['val_f1'].index(best_s2_f1) + 1

    #  Train vs Val Loss 
    fig, ax = plt.subplots(figsize=(10, 5))
    fig.patch.set_facecolor(BG)
    _style_ax(ax)
    ax.plot(s2_epochs, s2_history['train_loss'], label='Train Loss',
            color='#2563eb', linewidth=2)
    ax.plot(s2_epochs, s2_history['val_loss'],   label='Val Loss',
            color='#dc2626', linewidth=2)
    ax.set_title('S2 Train vs Val Loss', fontsize=13)
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
    ax.legend(fontsize=10, framealpha=0.9, loc='upper right')
    ax.set_ylim(bottom=0)
    _save_show(fig, 's2_c1_loss.png')

    # Train vs Val F1 
    fig, ax = plt.subplots(figsize=(10, 5))
    fig.patch.set_facecolor(BG)
    _style_ax(ax)
    ax.plot(s2_epochs, s2_history['train_f1'], label='Train F1',
            color='#2563eb', linewidth=2)
    ax.plot(s2_epochs, s2_history['val_f1'],   label='Val F1',
            color='#dc2626', linewidth=2)
    _best_vline(ax, best_s2_ep, 'Best F1 Epoch')
    ax.set_title('S2 F1 Score', fontsize=13)
    ax.set_xlabel('Epoch'); ax.set_ylabel('F1')
    best_handle = Line2D([0], [0], color='#16a34a', linestyle=':', linewidth=2,
                         label=f'Best Val F1={best_s2_f1:.4f} (Ep {best_s2_ep})')
    h, l = ax.get_legend_handles_labels()
    ax.legend(handles=h + [best_handle], fontsize=9,
              framealpha=0.9, loc='lower right')
    _save_show(fig, 's2_c2_f1.png')

    # Train vs Val IoU 
    fig, ax = plt.subplots(figsize=(10, 5))
    fig.patch.set_facecolor(BG)
    _style_ax(ax)
    ax.plot(s2_epochs, s2_history['train_iou'], label='Train IoU',
            color='#0891b2', linewidth=2)
    ax.plot(s2_epochs, s2_history['val_iou'],   label='Val IoU',
            color='#e11d48', linewidth=2)
    ax.set_title('S2 IoU Score', fontsize=13)
    ax.set_xlabel('Epoch'); ax.set_ylabel('IoU')
    ax.legend(fontsize=10, framealpha=0.9, loc='lower right')
    _save_show(fig, 's2_c3_iou.png')

    # Train vs Val Precision 
    fig, ax = plt.subplots(figsize=(10, 5))
    fig.patch.set_facecolor(BG)
    _style_ax(ax)
    ax.plot(s2_epochs, s2_history['train_prec'], label='Train Precision',
            color='#2563eb', linewidth=2)
    ax.plot(s2_epochs, s2_history['val_prec'],   label='Val Precision',
            color='#dc2626', linewidth=2)
    ax.set_title('S2 Precision', fontsize=13)
    ax.set_xlabel('Epoch'); ax.set_ylabel('Precision')
    ax.legend(fontsize=10, framealpha=0.9, loc='lower right')
    _save_show(fig, 's2_c4_precision.png')

    # Train vs Val Recall 
    fig, ax = plt.subplots(figsize=(10, 5))
    fig.patch.set_facecolor(BG)
    _style_ax(ax)
    ax.plot(s2_epochs, s2_history['train_rec'], label='Train Recall',
            color='#0891b2', linewidth=2)
    ax.plot(s2_epochs, s2_history['val_rec'],   label='Val Recall',
            color='#e11d48', linewidth=2)
    ax.set_title('S2 Recall', fontsize=13)
    ax.set_xlabel('Epoch'); ax.set_ylabel('Recall')
    ax.legend(fontsize=10, framealpha=0.9, loc='lower right')
    _save_show(fig, 's2_c5_recall.png')

In [ ]:
if EVAL_S2:
    s2_ckpt = MODEL_PATH_S2          
    model_s2.load_state_dict(torch.load(s2_ckpt, map_location=device))
    model_s2.eval()
    
    t_tp = t_tn = t_fp = t_fn = 0

    flip_pairs = [
        (lambda x: x,          lambda x: x),           
        (TF.hflip,              TF.hflip),             
        (TF.vflip,              TF.vflip),             
        (lambda x: TF.hflip(TF.vflip(x)),
         lambda x: TF.hflip(TF.vflip(x))),            
    ]


    total_test_start = time.time()
    total_img_pairs = len(test_loader.dataset)
    
    with torch.no_grad():
        for img1, img2, mask in test_loader:
            img1, img2, mask = img1.to(device), img2.to(device), mask.to(device)
    
            p2_preds = []
    
            for flip_in, flip_out in flip_pairs:
                i1 = torch.stack([flip_in(img1[b]) for b in range(img1.shape[0])])
                i2 = torch.stack([flip_in(img2[b]) for b in range(img2.shape[0])])
    
                with torch.amp.autocast('cuda'):
                    p1_f, d4_f, d3_f, d2_f = model_s1.forward_with_stage2_features(i1, i2)
                    p2_f, _, _ = model_s2(d3_f, d4_f, d2_f, p1_f)
    
                p2_sig = torch.sigmoid(p2_f)
    
                p2_preds.append(
                    torch.stack([flip_out(p2_sig[b]) for b in range(p2_sig.shape[0])])
                )
    
            p2_avg = torch.mean(torch.stack(p2_preds), dim=0)
    
            pred_bin = (p2_avg > 0.5).float()
            tp = (pred_bin * mask).sum().float()
            tn = ((1 - pred_bin) * (1 - mask)).sum().float()
            fp = (pred_bin * (1 - mask)).sum().float()
            fn = ((1 - pred_bin) * mask).sum().float()
    
            t_tp += tp
            t_tn += tn
            t_fp += fp
            t_fn += fn

    total_test_time_2 = time.time() - total_test_start
    time_per_img_pair_2 = total_test_time_2 / total_img_pairs
    
    s2_test = finalize_metrics(t_tp, t_tn, t_fp, t_fn)
    
    

    print("\n" + "═" * 50)
    print("  STAGE 2 Test Metrics")
    print("═" * 50)

    print(f"  Dataset              : {DATASET}")
    print(f"  Total image pairs    : {total_img_pairs}")
    print(f"  Total test time      : {total_test_time_2:.2f} sec")
    print(f"  Time / image pair    : {time_per_img_pair_2:.4f} sec")
    
    print("=" * 56)
    print(f"  Accuracy  : {s2_test['acc']:.4f}")
    print(f"  Precision : {s2_test['prec']:.4f}")
    print(f"  Recall    : {s2_test['rec']:.4f}")
    print(f"  F1 Score  : {s2_test['f1']:.4f}")
    print(f"  IoU       : {s2_test['iou']:.4f}") 

    print("═" * 50)

In [ ]:
if EVAL_S1 and EVAL_S2:
    final_total_time = total_test_time + total_test_time_2
    final_time_per_image = final_total_time / total_img_pairs

    print("\n" + "=" * 68)
    print("  STAGE 1 vs STAGE 2 — Final Test Comparison")
    print("=" * 68)

    print(f"  Dataset               : {DATASET}")
    print(f"  Total image pairs     : {total_img_pairs}")

    print("-" * 68)
    print(f"  Stage 1 total time    : {total_test_time:.2f} sec")
    print(f"  Stage 1 / image pair  : {time_per_img_pair:.4f} sec")

    print(f"  Stage 2 total time    : {total_test_time_2:.2f} sec")
    print(f"  Stage 2 / image pair  : {time_per_img_pair_2:.4f} sec")

    print("-" * 68)
    print(f"  Final total time      : {final_total_time:.2f} sec")
    print(f"  Final / image pair    : {final_time_per_image:.4f} sec")

    print("=" * 68)
    print(f"  {'Metric':<12} | {'Stage 1':>8} | {'Stage 2':>8} | {'Change':>8}")
    print(f"  {'-'*12}-+-{'-'*8}-+-{'-'*8}-+-{'-'*8}")

    for k in ['acc', 'prec', 'rec', 'f1', 'iou']:
        delta = s2_test[k] - s1_test[k]
        sign = "+" if delta >= 0 else ""
        print(f"  {k:<12} | {s1_test[k]:>8.4f} | {s2_test[k]:>8.4f} | {sign}{delta:.4f}")

    print("=" * 68)

Display

In [ ]:
if EVAL_S2:
    
    # Config

    N_SAMPLES        = 3      
    DISPLAY_THRESHOLD = 0.50   # raise to 0.60–0.70 if too many FP shown
    MIN_BBOX_AREA    = 80      #  min pixel area for a bbox to appear
    BBOX_SEPARATE    = True    
    ALPHA_OVERLAY    = 0.55    # mask transparency on overlay
    SAVE_FIG         = True
    
    MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
    STD  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
    
    def denorm(t):
        img = (t.cpu() * STD + MEAN).clamp(0,1)
        return (img.permute(1,2,0).numpy() * 255).astype(np.uint8)
    
    prob_cmap = LinearSegmentedColormap.from_list(
        "prob", ["#050014","#1a0050","#5b0ea6","#c0006e","#ff4500","#ffc500"], N=256)
    
    model_s1.eval(); model_s2.eval()
    samples = []
    
    with torch.no_grad():
        for img1_b, img2_b, mask_b in test_loader:
            img1_b = img1_b.to(device)
            img2_b = img2_b.to(device)
            mask_b = mask_b.to(device)
            B = img1_b.shape[0]
            for i in range(B):
                if len(samples) >= N_SAMPLES:
                    break
                with torch.amp.autocast('cuda'):
                    p1, diff4, diff3_attn, diff2_raw = \
                        model_s1.forward_with_stage2_features(
                            img1_b[i:i+1], img2_b[i:i+1])
                    p2, _,_ = model_s2(diff3_attn, diff4, diff2_raw, p1)
    
                prob  = torch.sigmoid(p2)[0,0].cpu().numpy()   # [H,W] 0-1
                pred  = (prob > DISPLAY_THRESHOLD).astype(np.uint8)
                gt    = mask_b[i,0].cpu().numpy().astype(np.uint8)
    
                if gt.sum() < 100 and len(samples) < N_SAMPLES - 1:
                    continue
    
                samples.append({
                    "img1": denorm(img1_b[i].cpu()),
                    "img2": denorm(img2_b[i].cpu()),
                    "prob": prob,
                    "pred": pred,
                    "gt"  : gt,
                })
            if len(samples) >= N_SAMPLES:
                break
    
    def make_clean_overlay(img_rgb, binary_mask, color, alpha=ALPHA_OVERLAY):
        result = img_rgb.copy().astype(np.float32)
        c = np.array(color, dtype=np.float32)
        px = binary_mask.astype(bool)
        result[px] = result[px] * (1 - alpha) + c * alpha
        return result.clip(0, 255).astype(np.uint8)
    
    
    def draw_separated_bboxes(img_rgb, pred_bin,
                               min_area=MIN_BBOX_AREA,
                               separate=BBOX_SEPARATE):
      
        canvas = img_rgb.copy()
    
        if separate:
            kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
            separated = cv2.erode(pred_bin, kernel, iterations=2)
        else:
            separated = pred_bin
    
        n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
            separated, connectivity=8)
    
        for j in range(1, n_labels):
            area = stats[j, cv2.CC_STAT_AREA]
            if area < min_area:
                continue
            x = stats[j, cv2.CC_STAT_LEFT]
            y = stats[j, cv2.CC_STAT_TOP]
            w = stats[j, cv2.CC_STAT_WIDTH]
            h = stats[j, cv2.CC_STAT_HEIGHT]
            # Skip boxes that fill >60% of the image (likely FP flood)
            if w * h > 0.60 * pred_bin.shape[0] * pred_bin.shape[1]:
                continue
            # Tight colored box
            cv2.rectangle(canvas, (x, y), (x+w, y+h),
                          color=(255, 100, 0), thickness=2)
            # Small area label
            cv2.putText(canvas, f"{area}px", (x+2, max(y-4, 10)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.32, (255,200,0), 1,
                        cv2.LINE_AA)
        return canvas
    
    #  columns per sample 
    #  Col 0: T1
    #  Col 1: T2
    #  Col 2: Probability heatmap
    #  Col 3: T2 + change overlay (red = predicted change only)
    #  Col 4: T2 + separated bounding boxes
    #  Col 5: T2 + GT overlay (gold = actual change)
    
    COL_TITLES = ["T1  (Before)", "T2  (After)",
                   "Prob Heatmap",
                   f"Predicted Change\n(thresh={DISPLAY_THRESHOLD})",
                   "Object Bboxes",
                   "Ground Truth"]
    
    COLS = 6
    ROWS = N_SAMPLES
    fig = plt.figure(figsize=(COLS * 3.5, ROWS * 3.5 + 0.9),
                     facecolor="#0d0d1a")
    gs  = gridspec.GridSpec(ROWS, COLS,
                             figure=fig,
                             hspace=0.06, wspace=0.04,
                             top=0.94, bottom=0.06)
    
    for c, title in enumerate(COL_TITLES):
        ax = fig.add_subplot(gs[0, c])
        ax.set_title(title, color="white", fontsize=10,
                      fontweight="bold", pad=5,
                      multialignment="center")
        ax.axis("off")
    
    for r, s in enumerate(samples):
        pred_overlay = make_clean_overlay(
            s["img2"], s["pred"],
            color=[220, 30, 30])          # red = predicted change
    
        gt_overlay   = make_clean_overlay(
            s["img2"], s["gt"],
            color=[255, 210, 0])          # gold = ground truth change
    
        bbox_img     = draw_separated_bboxes(s["img2"], s["pred"])
    
        # Build contour outline for change mask (shows boundary, not flood fill)
        contours, _ = cv2.findContours(
            s["pred"], cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        contour_img  = s["img2"].copy()
        cv2.drawContours(contour_img, contours, -1, (255, 60, 60), 2)
    
        panels = [
            (s["img1"],    None,      {}),
            (s["img2"],    None,      {}),
            (s["prob"],    prob_cmap, {"vmin": 0, "vmax": 1}),
            (pred_overlay, None,      {}),
            (bbox_img,     None,      {}),
            (gt_overlay,   None,      {}),
        ]
    
        for c_idx, (data, cmap, kwargs) in enumerate(panels):
            ax = fig.add_subplot(gs[r, c_idx])
            ax.set_facecolor("#0d0d1a")
            ax.set_xticks([]); ax.set_yticks([])
            for sp in ax.spines.values():
                sp.set_edgecolor("#2a2a4a"); sp.set_linewidth(0.6)
    
            if cmap:
                im = ax.imshow(data, cmap=cmap, **kwargs)
                if c_idx == 2:   # prob map colorbar
                    cb = plt.colorbar(im, ax=ax,
                                       fraction=0.045, pad=0.02,
                                       ticks=[0, 0.25, 0.5, 0.75, 1.0])
                    cb.ax.tick_params(colors="white", labelsize=6.5)
                    cb.outline.set_edgecolor("#2a2a4a")
                    cb.set_label("P(change)", color="white",
                                  fontsize=7, labelpad=3)
            else:
                ax.imshow(data)
    
            # Change pixel count annotation
            if c_idx == 3:
                n_px = int(s["pred"].sum())
                pct  = 100 * n_px / (s["pred"].shape[0] * s["pred"].shape[1])
                ax.set_xlabel(f"Changed: {n_px}px  ({pct:.1f}%)",
                               color="#ff9999", fontsize=7.5)
            if c_idx == 5:
                n_gt = int(s["gt"].sum())
                pct_gt = 100 * n_gt / (s["gt"].shape[0] * s["gt"].shape[1])
                ax.set_xlabel(f"GT change: {n_gt}px  ({pct_gt:.1f}%)",
                               color="#ffd700", fontsize=7.5)
    
            if c_idx == 0:
                ax.set_ylabel(f"Sample {r+1}",
                               color="#aaaacc", fontsize=9, labelpad=4)
    
    # ── Legend ───────────────────────────────────────────────────────────────
    legend_ax = fig.add_axes([0.02, 0.005, 0.96, 0.04])
    legend_ax.set_facecolor("#0d0d1a"); legend_ax.axis("off")
    patches = [
        mpatches.Patch(color="#dc1e1e", label="Predicted Change (overlay)"),
        mpatches.Patch(color="#ffd700", label="Ground Truth Change"),
        mpatches.Patch(color=(1.0, 0.39, 0.0), label="Bounding Box"),
        mpatches.Patch(color="#ffc500", label="High Prob (→1.0)"),
        mpatches.Patch(color="#050014", label="Low Prob  (→0.0)"),
    ]
    legend_ax.legend(handles=patches, loc="center", ncol=5,
                      frameon=False, labelcolor="white",
                      fontsize=9, handlelength=1.4)
    
    fig.suptitle(
        "Stage 2 Change Detection — Output Visualization  "
        f"(threshold={DISPLAY_THRESHOLD})",
        color="white", fontsize=13, fontweight="bold")
    
    if SAVE_FIG:
        fig.savefig("stage2_visualization.png", dpi=160,
                     bbox_inches="tight", facecolor="#0d0d1a")
        print("Saved → stage2_visualization.png")
    
    plt.show()
    
    